# Chapter 4: Data Wrangling with Pandas & DuckDB SQL
### *Modern Machine Learning & Data Science Engineering* by Vaibhav Rastogi

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/v893979/Modern_Machine_Learning_and_Data_Science_Engineering/blob/main/src/ch04_pandas_and_duckdb.ipynb)

---

## 📌 Notebook Overview
Wrangle credit portfolio tabular data with Pandas and execute embedded in-memory analytical SQL queries using DuckDB.

In [ ]:
# Environment & Data Auto-Download for Google Colab
import os
import urllib.request

os.makedirs('../data', exist_ok=True)
data_path = "../data/credit_risk_dataset.csv"
if not os.path.exists(data_path):
    url = "https://raw.githubusercontent.com/v893979/Modern_Machine_Learning_and_Data_Science_Engineering/main/data/credit_risk_dataset.csv"
    print(f"Downloading credit_risk_dataset.csv from GitHub...")
    urllib.request.urlretrieve(url, data_path)
    print("Download complete!")


### 1. Ingestion & GroupBy Aggregations in Pandas

Load 1,000 credit records and compute risk aggregations by loan intent:

In [ ]:
import pandas as pd

df = pd.read_csv("../data/credit_risk_dataset.csv")
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")

summary = df.groupby("loan_intent").agg(
    avg_income=("person_income", "mean"),
    avg_loan=("loan_amnt", "mean"),
    default_rate=("loan_status", "mean"),
    applicant_count=("loan_status", "count")
).reset_index()
display(summary)

### 2. Embedded High-Speed SQL with DuckDB

Query the Pandas DataFrame directly with standard SQL window functions:

In [ ]:
!pip install -q duckdb
import duckdb

query = """
SELECT loan_intent,
       COUNT(*) as total_applicants,
       ROUND(AVG(loan_amnt), 2) as mean_loan_usd,
       ROUND(AVG(loan_status) * 100, 2) as default_rate_pct
FROM df
GROUP BY loan_intent
ORDER BY default_rate_pct DESC
"""
sql_result = duckdb.query(query).to_df()
display(sql_result)